In [ ]:
import json
import logging
from typing import Dict, Any, List

import wandb
import weave
from datasets import Dataset

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

from unsloth.chat_templates import train_on_responses_only


MODEL_NAME = "unsloth/Qwen3-4B-bnb-4bit"
DATA_PATH = "data/insta-sft.jsonl"
MAX_SEQ_LENGTH = 2048


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)


def init_wandb():
    wandb.init(project="gur-prime", name="qlora-run", reinit=True)


def init_weave():
    weave.init("gur-prime")


def validate_message(msg: Dict[str, Any]) -> bool:
    if not isinstance(msg, dict):
        return False
    if "role" not in msg or "content" not in msg:
        return False
    if not isinstance(msg["role"], str) or not isinstance(msg["content"], str):
        return False
    if len(msg["content"].strip()) == 0:
        return False
    return True


def validate_example(example: Dict[str, Any], idx: int) -> Dict[str, Any]:
    if "messages" not in example:
        raise ValueError(f"Row {idx}: missing 'messages' field")

    messages = example["messages"]

    if not isinstance(messages, list):
        raise ValueError(f"Row {idx}: messages is not a list")

    if len(messages) < 2:
        raise ValueError(f"Row {idx}: messages too short")

    cleaned = []

    for m in messages:
        if not validate_message(m):
            raise ValueError(f"Row {idx}: invalid message structure: {m}")

        cleaned.append(
            {
                "role": m["role"].strip(),
                "content": m["content"].strip(),
            }
        )

    return {"messages": cleaned}


def load_and_validate_jsonl(path: str) -> List[Dict[str, Any]]:
    data = []

    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            try:
                obj = json.loads(line)
            except Exception as e:
                raise ValueError(f"Invalid JSON at line {i}: {e}")

            validated = validate_example(obj, i)
            data.append(validated)

    logger.info(f"Loaded and validated {len(data)} rows")
    return data


def load_dataset_safe():
    data = load_and_validate_jsonl(DATA_PATH)
    return Dataset.from_list(data)


def load_model():
    return FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )


def format_chat(example, tokenizer):
    try:
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )

        if not isinstance(text, str) or len(text.strip()) == 0:
            raise ValueError("Empty chat template output")

        return {"text": text.strip()}

    except Exception as e:
        raise ValueError(f"Chat formatting failed: {e}")


def main():
    try:
        init_wandb()
        init_weave()

        dataset = load_dataset_safe()

        dataset = dataset.train_test_split(test_size=0.1, seed=42)
        train_ds = dataset["train"]
        eval_ds = dataset["test"]
        logger.info(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")

        model, tokenizer = load_model()

        def format_chat(example):
            text = tokenizer.apply_chat_template(
                example["messages"],
                tokenize=False,
                add_generation_prompt=False,
            )
            return {"text": text.strip()}

        train_ds = train_ds.map(format_chat)
        eval_ds = eval_ds.map(format_chat)

        train_ds = train_ds.filter(lambda x: len(x.get("text", "").strip()) > 0)
        eval_ds = eval_ds.filter(lambda x: len(x.get("text", "").strip()) > 0)

        model = FastLanguageModel.get_peft_model(
            model,
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
                "gate_proj",
                "up_proj",
                "down_proj",
            ],
            use_gradient_checkpointing="unsloth",
        )

        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LENGTH,
            args=SFTConfig(
                per_device_train_batch_size=2,
                gradient_accumulation_steps=4,
                warmup_steps=10,
                num_train_epochs=2,
                learning_rate=2e-4,
                logging_steps=1,
                eval_strategy="steps",
                eval_steps=20,
                save_strategy="steps",
                save_steps=20,
                load_best_model_at_end=True,
                metric_for_best_model="eval_loss",
                optim="adamw_8bit",
                lr_scheduler_type="linear",
                report_to="wandb",
                output_dir="outputs",
                seed=42,
            ),
        )

        trainer = train_on_responses_only(
            trainer,
            instruction_part="<|im_start|>user\n",
            response_part="<|im_start|>assistant\n",
        )

        trainer.train()

        model.save_pretrained("outputs/best_lora")
        tokenizer.save_pretrained("outputs/best_lora")
        logger.info("Saved best LoRA adapter")

    except Exception as e:
        logger.exception("Training pipeline failed")
        raise


if __name__ == "__main__":
    main()